# Lab 9 — Word Embeddings (Word2Vec / FastText)

Track A corpus analysis on cleaned `processed_v2` with a direct comparison of Word2Vec and FastText nearest neighbors.

## 1. Install deps

In [1]:
from pathlib import Path
import subprocess
import sys

if Path('/content').exists() and not Path('/content/nlp_labs').exists():
    subprocess.run(['git', 'clone', 'https://github.com/velotsuraptor/nlp_labs.git', '/content/nlp_labs'], check=True)

project_root_candidates = [
    Path('/content/nlp_labs/project_lab9'),
    Path.cwd(),
    Path.cwd().parent,
    Path.cwd().parent.parent,
]
PROJECT_ROOT = None
for cand in project_root_candidates:
    if (cand / 'requirements.txt').exists() and (cand / 'src').exists():
        PROJECT_ROOT = cand
        break
    if (cand / 'project_lab9' / 'requirements.txt').exists():
        PROJECT_ROOT = cand / 'project_lab9'
        break
if PROJECT_ROOT is None:
    raise FileNotFoundError('Could not locate project_lab9/requirements.txt')

REPO_ROOT = PROJECT_ROOT.parent
req_path = PROJECT_ROOT / 'requirements.txt'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(req_path)], check=True)
print('project_root =', PROJECT_ROOT)

project_root = C:\Users\maia1\data\politiekh\masters\nlp\project_lab9


## 2. Data access

In [2]:
from collections import Counter
from pathlib import Path
import sys
import pandas as pd

ROOT = PROJECT_ROOT
sys.path.insert(0, str(REPO_ROOT))

from project_lab9.src.embeddings_train import (
    EmbeddingConfig,
    load_embedding_corpus,
    prepare_sentences,
    tokenize_for_embeddings,
    train_fasttext,
    train_word2vec,
)
from project_lab9.src.embeddings_eval import (
    build_case_summary,
    build_domain_terms_table,
    build_neighbors_table,
    compare_models_for_word,
    format_neighbors,
    safe_neighbors,
)

raw_df = load_embedding_corpus(ROOT)
raw_df.head(3)

,text_id,text,sentences,label
0,9905,"Вступив на ІСТ цього року, тепер молюся, щоб п...","[""Вступив на ІСТ цього року, тепер молюся, щоб...",Question / Request for Help
1,10001201,Цифрова держава Повідомлення 123 від 18.04.202...,"[""Цифрова держава Повідомлення 123 від 18.04.2...",Question / Request for Help
2,3099,Старий університет поки що вчить. Наразі налаш...,"[""Старий університет поки що вчить."", ""Наразі ...",Neutral Comment


## 3. Corpus preparation

In [3]:
TEXT_COL = 'text'
sentences = prepare_sentences(raw_df, text_col=TEXT_COL)
token_count = sum(len(s) for s in sentences)
doc_count = len(raw_df)
kept_docs = len(sentences)
print('text_field =', TEXT_COL)
print('documents =', doc_count)
print('documents with at least one token =', kept_docs)
print('approx_token_count =', token_count)

token_counter = Counter()
for sent in sentences:
    token_counter.update(sent)
pd.DataFrame(token_counter.most_common(20), columns=['token', 'count'])

text_field = text
documents = 1000
documents with at least one token = 1000
approx_token_count = 22030


,token,count
0,на,562
1,не,419
2,в,376
3,і,372
4,у,299
5,та,289
6,з,271
7,що,241
8,за,179
9,до,175


## 4. Tokenization check

In [4]:
tokenization_df = raw_df[['text_id', 'text']].head(5).copy()
tokenization_df['tokens'] = tokenization_df['text'].map(tokenize_for_embeddings)
tokenization_df

,text_id,text,tokens
0,9905,"Вступив на ІСТ цього року, тепер молюся, щоб п...","[вступив, на, іст, цього, року, тепер, молюся,..."
1,10001201,Цифрова держава Повідомлення 123 від 18.04.202...,"[цифрова, держава, повідомлення, від, року, на..."
2,3099,Старий університет поки що вчить. Наразі налаш...,"[старий, університет, поки, що, вчить, наразі,..."
3,8664,"На пл. Ринок ЦНАП м.Львова, швидке ьа якісне в...","[на, пл, ринок, цнап, м, львова, швидке, ьа, я..."
4,1035,"Мені здається, що наша кузня супер-кадрів в IT...","[мені, здається, що, наша, кузня, супер-кадрів..."


## 5. Train Word2Vec

In [5]:
cfg = EmbeddingConfig(vector_size=100, window=5, min_count=3, sg=1, epochs=20, seed=42, workers=1)
w2v_model = train_word2vec(sentences, cfg)
print('Word2Vec vocab size =', len(w2v_model.wv))
print('Word2Vec config =', cfg)

Word2Vec vocab size = 1373
Word2Vec config = EmbeddingConfig(vector_size=100, window=5, min_count=3, sg=1, epochs=20, seed=42, workers=1)


## 6. Train FastText

In [6]:
fasttext_model = train_fasttext(sentences, cfg)
print('FastText vocab size =', len(fasttext_model.wv.key_to_index))
print('FastText config =', cfg)

FastText vocab size = 1373
FastText config = EmbeddingConfig(vector_size=100, window=5, min_count=3, sg=1, epochs=20, seed=42, workers=1)


## 7. Nearest neighbors analysis

In [7]:
analysis_words = [
    ('університет', 'frequent/domain'),
    ('євідновлення', 'domain'),
    ('паспорт', 'domain'),
    ('дія', 'domain/morph-sensitive'),
    ('ремонт', 'domain'),
    ('ратуша', 'rare/domain'),
    ('черга', 'morph-variant'),
    ('реєстрація', 'rare'),
    ('phone', 'noisy/latin'),
    ('oкyпації', 'noisy/mixed-script'),
]

usefulness_map = {
    'університет': 'useful',
    'євідновлення': 'useful',
    'паспорт': 'partly',
    'дія': 'useful',
    'ремонт': 'partly',
    'ратуша': 'useful',
    'черга': 'partly',
    'реєстрація': 'weak',
    'phone': 'weak',
    'oкyпації': 'weak',
}
comment_map = {
    'університет': 'FastText groups morphology better; Word2Vec is also coherent on frequent forms.',
    'євідновлення': 'Both models catch the program context; FastText adds cleaner morphological neighbors.',
    'паспорт': 'Domain signal is present, but Word2Vec pulls some noisy context words.',
    'дія': 'Both models stay in the e-service / damaged-property context, with cleaner FastText neighborhoods.',
    'ремонт': 'Neighbors are only partly useful; the term drifts toward adjacent administrative or housing context.',
    'ратуша': 'FastText is clearly better on morphology and landmark-specific neighbors.',
    'черга': 'Both models see the queue/waiting concept, while FastText captures inflected variants better.',
    'реєстрація': 'Word2Vec is unstable for this rare word; FastText is better but still not reliably semantic.',
    'phone': 'Latin/noisy token gives weak or accidental neighbors in both models.',
    'oкyпації': 'Word2Vec is OOV; FastText can compose a vector but the neighbors are not practically useful.',
}

neighbors_table = build_neighbors_table(
    analysis_words,
    w2v_model,
    fasttext_model,
    usefulness_map=usefulness_map,
    comment_map=comment_map,
    topn=8,
)
neighbors_table

,Word,Type,Word2Vec neighbors,FastText neighbors,Useful?,Comment
0,університет,frequent/domain,"університету (0.933), імені (0.923), площі (0....","університету (0.998), університетів (0.994), у...",useful,FastText groups morphology better; Word2Vec is...
1,євідновлення,domain,"майно (0.973), заяву (0.972), подати (0.972), ...","відновлення (0.997), оновлення (0.993), пошкод...",useful,Both models catch the program context; FastTex...
2,паспорт,domain,"прийняли (0.943), документи (0.939), без (0.93...","паспорта (0.987), паспорти (0.985), документ (...",partly,"Domain signal is present, but Word2Vec pulls s..."
3,дія,domain/morph-sensitive,"пошкоджене (0.977), застосунку (0.977), майно ...","заяви (0.996), пошкоджене (0.994), заяву (0.99...",useful,Both models stay in the e-service / damaged-pr...
4,ремонт,domain,"пошкодженого (0.947), отримати (0.937), окрім ...","ремонту (0.990), податків (0.984), подавав (0....",partly,Neighbors are only partly useful; the term dri...
5,ратуша,rare/domain,"ринок (0.978), площі (0.977), зали (0.974), са...","верх (0.996), ратуші (0.996), вершини (0.994),...",useful,FastText is clearly better on morphology and l...
6,черга,morph-variant,"черзі (0.964), очікування (0.954), години (0.9...","чергу (0.995), черзі (0.991), черги (0.980), ч...",partly,"Both models see the queue/waiting concept, whi..."
7,реєстрація,rare,"пенсійного (0.994), знайти (0.992), додати (0....","нотаріуса (0.992), нотаріуси (0.990), запис (0...",weak,Word2Vec is unstable for this rare word; FastT...
8,phone,noisy/latin,"рр (0.875), резиденція (0.842), зв'язку (0.823...","рр (0.919), будинку (0.907), резиденція (0.907...",weak,Latin/noisy token gives weak or accidental nei...
9,oкyпації,noisy/mixed-script,[oov],"щодо (0.988), завдання (0.978), спілкування (0...",weak,Word2Vec is OOV; FastText can compose a vector...


## 8. Domain terms analysis

In [8]:
domain_terms = ['євідновлення', 'дія', 'паспорт', 'ремонт', 'ратуша']
domain_judgements = {
    'євідновлення': 'FastText is slightly better: both are useful, but FastText keeps more morphological/domain variants together.',
    'дія': 'FastText is better because it stays closer to applications, claims, and damaged-property processing vocabulary.',
    'паспорт': 'FastText is better on morphology; Word2Vec is noisier but still domain-related.',
    'ремонт': 'Both are only partly useful; the corpus is too mixed for a stable repair-specific neighborhood.',
    'ратуша': 'FastText is clearly better because subword information helps align ратуша / ратуші / ратушу / вежа.',
}
domain_table = build_domain_terms_table(domain_terms, w2v_model, fasttext_model, domain_judgements, topn=8)
domain_table

,term,word2vec_neighbors,fasttext_neighbors,judgement
0,євідновлення,"майно (0.973), заяву (0.972), подати (0.972), ...","відновлення (0.997), оновлення (0.993), пошкод...","FastText is slightly better: both are useful, ..."
1,дія,"пошкоджене (0.977), застосунку (0.977), майно ...","заяви (0.996), пошкоджене (0.994), заяву (0.99...",FastText is better because it stays closer to ...
2,паспорт,"прийняли (0.943), документи (0.939), без (0.93...","паспорта (0.987), паспорти (0.985), документ (...",FastText is better on morphology; Word2Vec is ...
3,ремонт,"пошкодженого (0.947), отримати (0.937), окрім ...","ремонту (0.990), податків (0.984), подавав (0....",Both are only partly useful; the corpus is too...
4,ратуша,"ринок (0.978), площі (0.977), зали (0.974), са...","верх (0.996), ратуші (0.996), вершини (0.994),...",FastText is clearly better because subword inf...


## 9. 5 “useful / not useful” cases

In [9]:
case_words = ['євідновлення', 'ратуша', 'черга', 'реєстрація', 'oкyпації']
case_labels = {
    'євідновлення': 'useful',
    'ратуша': 'useful',
    'черга': 'partly',
    'реєстрація': 'weak',
    'oкyпації': 'weak',
}
case_rationales = {
    'євідновлення': 'Useful: both models recover domain neighbors; FastText is cleaner on morphology and related administrative forms.',
    'ратуша': 'Useful: the landmark neighborhood is concrete, and FastText strongly benefits from inflected forms and subwords.',
    'черга': 'Partly useful: the queue/waiting concept is visible, but some neighbors remain generic rather than task-specific.',
    'реєстрація': 'Weak: the word is rare, so Word2Vec is unstable and FastText only partly rescues it.',
    'oкyпації': 'Weak: this noisy mixed-script form is OOV for Word2Vec and only weakly recoverable in FastText.',
}
case_table = build_case_summary(case_words, case_labels, case_rationales, w2v_model, fasttext_model, topn=8)
case_table

,word,label,word2vec_neighbors,fasttext_neighbors,rationale
0,євідновлення,useful,"майно (0.973), заяву (0.972), подати (0.972), ...","відновлення (0.997), оновлення (0.993), пошкод...",Useful: both models recover domain neighbors; ...
1,ратуша,useful,"ринок (0.978), площі (0.977), зали (0.974), са...","верх (0.996), ратуші (0.996), вершини (0.994),...","Useful: the landmark neighborhood is concrete,..."
2,черга,partly,"черзі (0.964), очікування (0.954), години (0.9...","чергу (0.995), черзі (0.991), черги (0.980), ч...",Partly useful: the queue/waiting concept is vi...
3,реєстрація,weak,"пенсійного (0.994), знайти (0.992), додати (0....","нотаріуса (0.992), нотаріуси (0.990), запис (0...","Weak: the word is rare, so Word2Vec is unstabl..."
4,oкyпації,weak,[oov],"щодо (0.988), завдання (0.978), спілкування (0...",Weak: this noisy mixed-script form is OOV for ...


## 10. Word2Vec vs FastText comparison

In [10]:
comparison_text = (
    'For this corpus, FastText is the better default embedding model. '
    'On frequent stable words like університет, both models are already useful, but FastText groups inflected forms more consistently. '
    'On domain terms such as євідновлення, дія, and ратуша, FastText keeps a cleaner neighborhood and handles morphology more naturally. '
    'The biggest difference appears on rare or noisy forms: Word2Vec fails on OOV-like words such as oкyпації, while FastText can still build a subword-based vector. '
    'That said, FastText does not magically solve corpus noise: for phone and реєстрація the neighbors are still weak or mixed. '
    'So embeddings are useful here mainly for vocabulary exploration and domain-term inspection, not as a guaranteed strong semantic resource on every rare token.'
)
print(comparison_text)

For this corpus, FastText is the better default embedding model. On frequent stable words like університет, both models are already useful, but FastText groups inflected forms more consistently. On domain terms such as євідновлення, дія, and ратуша, FastText keeps a cleaner neighborhood and handles morphology more naturally. The biggest difference appears on rare or noisy forms: Word2Vec fails on OOV-like words such as oкyпації, while FastText can still build a subword-based vector. That said, FastText does not magically solve corpus noise: for phone and реєстрація the neighbors are still weak or mixed. So embeddings are useful here mainly for vocabulary exploration and domain-term inspection, not as a guaranteed strong semantic resource on every rare token.


## 11. Generate docs/audit_summary_lab9.md

In [11]:
from pathlib import Path

docs_dir = ROOT / 'docs'
docs_dir.mkdir(parents=True, exist_ok=True)
(ROOT / 'labs' / 'lab09').mkdir(parents=True, exist_ok=True)

strongest_examples = ['євідновлення', 'ратуша', 'університет']
weakest_examples = ['реєстрація', 'phone', 'oкyпації']

summary_md = f'''# Audit summary — Lab9

1. Corpus: cleaned `processed_v2` from Lab2; {doc_count} documents and about {token_count} tokens after light tokenization.
2. Models trained: Word2Vec and FastText with shared parameters (`vector_size=100`, `window=5`, `min_count=3`, `sg=1`, `epochs=20`, `seed=42`).
3. Strongest nearest-neighbor examples: {', '.join(strongest_examples)}.
4. Weakest nearest-neighbor examples: {', '.join(weakest_examples)}.
5. Domain terms with meaningful neighborhoods: євідновлення, дія, ратуша, and partially паспорт.
6. Where FastText won: morphology, inflected forms, and OOV-like noisy words.
7. Where the gain was small: frequent stable words and mixed-context terms such as ремонт.
8. Final judgement: embeddings are useful here for vocabulary exploration and domain-term inspection; FastText is the better fit for this noisy mixed corpus.
'''
(docs_dir / 'audit_summary_lab9.md').write_text(summary_md, encoding='utf-8')

notes_lines = []
notes_lines.append('# Embedding notes — Lab9')
notes_lines.append('')
notes_lines.append('## 1. Corpus')
notes_lines.append('')
notes_lines.append('- Source: processed `text` field from `processed_v2`')
notes_lines.append(f'- Documents: {doc_count}')
notes_lines.append(f'- Approx tokens after tokenization: {token_count}')
notes_lines.append('')
notes_lines.append('## 2. Models')
notes_lines.append('')
notes_lines.append('- Word2Vec')
notes_lines.append('- FastText')
notes_lines.append('')
notes_lines.append('## 3. Parameters')
notes_lines.append('')
notes_lines.append(f'- vector_size={cfg.vector_size}, window={cfg.window}, min_count={cfg.min_count}, sg={cfg.sg}, epochs={cfg.epochs}, seed={cfg.seed}, workers={cfg.workers}')
notes_lines.append('')
notes_lines.append('## 4. Ten nearest-neighbor probe words')
notes_lines.append('')
for _, row in neighbors_table.iterrows():
    notes_lines.append(f'- {row["Word"]} [{row["Type"]}]')
    notes_lines.append(f'  W2V: {row["Word2Vec neighbors"]}')
    notes_lines.append(f'  FT: {row["FastText neighbors"]}')
    notes_lines.append(f'  Useful: {row["Useful?"]}; Comment: {row["Comment"]}')
notes_lines.append('')
notes_lines.append('## 5. Five domain terms')
notes_lines.append('')
for _, row in domain_table.iterrows():
    notes_lines.append(f'- {row["term"]}')
    notes_lines.append(f'  W2V: {row["word2vec_neighbors"]}')
    notes_lines.append(f'  FT: {row["fasttext_neighbors"]}')
    notes_lines.append(f'  Judgement: {row["judgement"]}')
notes_lines.append('')
notes_lines.append('## 6. Five useful / not useful cases')
notes_lines.append('')
for _, row in case_table.iterrows():
    notes_lines.append(f'- {row["word"]}: {row["label"]}')
    notes_lines.append(f'  W2V: {row["word2vec_neighbors"]}')
    notes_lines.append(f'  FT: {row["fasttext_neighbors"]}')
    notes_lines.append(f'  Rationale: {row["rationale"]}')
notes_lines.append('')
notes_lines.append('## 7. Word2Vec vs FastText')
notes_lines.append('')
notes_lines.append(comparison_text)
notes_lines.append('')
notes_lines.append('## 8. Main conclusion')
notes_lines.append('')
notes_lines.append('FastText is the more useful embedding model for this corpus because it handles morphology and noisy tokens better, although both models remain limited by corpus size and domain heterogeneity.')
(docs_dir / 'embedding_notes_lab9.md').write_text('\n'.join(notes_lines), encoding='utf-8')

dataset_card_md = f'''# Dataset card — Lab9

## Embeddings readiness
- Corpus used: `processed_v2` original word forms.
- Size for embeddings: {doc_count} documents and about {token_count} tokens after light tokenization.
- Domain vocabulary is present (`євідновлення`, `дія`, `паспорт`, `ремонт`, `ратуша`), but the corpus is still mixed across several subdomains.
- Noisy text exists: mixed-script tokens, Latin tokens, spelling variation, and short low-context texts.
- FastText looks more appropriate than Word2Vec for this corpus because morphology and subword composition matter in Ukrainian and in noisy user text.
- Embeddings provide a useful exploratory signal, especially for domain terms, but are not uniformly strong on rare or noisy words.
'''
(docs_dir / 'dataset_card.md').write_text(dataset_card_md, encoding='utf-8')

readme_md = f'''# LPNU NLP — Lab 09 (Word Embeddings: Word2Vec vs FastText)

1. Corpus: cleaned Ukrainian `processed_v2` comments/reviews using original word forms.
2. Models: Word2Vec and FastText with the same training parameters.
3. Parameters: `vector_size=100`, `window=5`, `min_count=3`, `sg=1`, `epochs=20`, `seed=42`.
4. Word types analyzed: frequent, rare, domain, morph-variant, noisy / Latin / mixed-script.
5. Most informative cases: `євідновлення`, `ратуша`, `черга`, `реєстрація`, `oкyпації`.
6. Where FastText was better: morphology, inflection, and OOV-like noisy variants.
7. Usefulness: embeddings are useful for vocabulary and domain-term analysis, with FastText as the better default on this corpus.
'''
(ROOT / 'labs' / 'lab09' / 'README.md').write_text(readme_md, encoding='utf-8')

print((docs_dir / 'audit_summary_lab9.md').read_text(encoding='utf-8'))

# Audit summary — Lab9

1. Corpus: cleaned `processed_v2` from Lab2; 1000 documents and about 22030 tokens after light tokenization.
2. Models trained: Word2Vec and FastText with shared parameters (`vector_size=100`, `window=5`, `min_count=3`, `sg=1`, `epochs=20`, `seed=42`).
3. Strongest nearest-neighbor examples: євідновлення, ратуша, університет.
4. Weakest nearest-neighbor examples: реєстрація, phone, oкyпації.
5. Domain terms with meaningful neighborhoods: євідновлення, дія, ратуша, and partially паспорт.
6. Where FastText won: morphology, inflected forms, and OOV-like noisy words.
7. Where the gain was small: frequent stable words and mixed-context terms such as ремонт.
8. Final judgement: embeddings are useful here for vocabulary exploration and domain-term inspection; FastText is the better fit for this noisy mixed corpus.

